<a href="https://colab.research.google.com/github/anirudhkartikeya/aisystems/blob/main/playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playground: ETH/UCY pedestrian trajectories (stand-in dataset)

This notebook loads and checks a real pedestrian trajectory dataset. The PyBullet simulator is not built yet, so this dataset stands in for the trajectories the simulator will produce.

## Data source
- **Dataset:** ETH/UCY pedestrian trajectories (ETH, Hotel, Univ, Zara1, Zara2), preprocessed
- **Downloaded from:** https://github.com/InhwanBae/NPSN (folder `dataset/`)
- **Version:** commit (paste the 7-character commit ID here)
- **Retrieved:** Oct 5, 2026
- **Original sources:** ETH (BIWI Walking Pedestrians, ETH Zurich): https://data.vision.ee.ethz.ch/cvl/aem/ewap_dataset_full.tgz. UCY (University of Cyprus crowds data): https://graphics.cs.ucy.ac.cy/research/downloads/crowd-data. The repo redistributes preprocessed copies, with the train/val/test splits used by Social-GAN.
- **Storage:** Google Drive, folder pybullet-nav-data/eth_ucy/raw/dataset (about 19 MB)
- **Personal data:** none (anonymous x/y coordinates only)

## License
- **Repo code license:** MIT License (Copyright (c) 2022 Inhwan Bae). This covers the code in the repo.
- **Data terms:** no explicit data license found in the repo. The original ETH and UCY pages were not checked for usage terms. Because redistribution terms are unconfirmed, the data is kept out of the GitHub repo and stored on Drive.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os
DATA_DIR = '/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw'
for root, dirs, files in os.walk(DATA_DIR):
    print(root, files[:5])

/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw []
/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset []
/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/raw []
/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/raw/train ['crowds_zara01_train.txt', 'crowds_zara03_train.txt', 'students001_train.txt', 'biwi_eth_train.txt', 'students003_train.txt']
/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/raw/val ['biwi_hotel_val.txt', 'students003_val.txt', 'biwi_eth_val.txt', 'uni_examples_val.txt', 'crowds_zara02_val.txt']
/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/raw/all_data ['crowds_zara01.txt', 'uni_examples.txt', 'biwi_eth.txt', 'crowds_zara02.txt', 'students001.txt']
/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/eth []
/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/eth/val ['crowds_zara01_val.txt', 'uni_examples_val.txt', 'students003_val.txt', 'crowds_zara03_val.txt', 'students001_val.txt']


In [14]:
import os
DATA_DIR = '/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/raw/all_data'
for name in sorted(os.listdir(DATA_DIR)):
    size_kb = os.path.getsize(os.path.join(DATA_DIR, name)) / 1024
    print(f"{name}: {size_kb:.1f} KB")

biwi_eth.txt: 114.2 KB
biwi_hotel.txt: 140.4 KB
crowds_zara01.txt: 200.9 KB
crowds_zara02.txt: 383.0 KB
crowds_zara03.txt: 195.2 KB
students001.txt: 859.1 KB
students003.txt: 707.3 KB
uni_examples.txt: 106.7 KB


In [15]:
with open(os.path.join(DATA_DIR, 'biwi_eth.txt')) as f:
    for _ in range(5):
        print(repr(f.readline()))

'780\t1.0\t8.46\t3.59\n'
'790\t1.0\t9.57\t3.79\n'
'800\t1.0\t10.67\t3.99\n'
'800\t2.0\t13.64\t5.8\n'
'810\t1.0\t11.73\t4.32\n'


In [16]:
!python --version

Python 3.13.16


In [17]:
import pandas, numpy
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)

pandas 2.2.3
numpy 2.1.3


In [18]:
import os
print(os.path.exists('/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/raw/all_data'))

True


In [21]:
# Mount Drive, load all recordings into one table, and show basic info
import os
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/pybullet-nav-data/eth_ucy/raw/dataset/raw/all_data'
print("Files found:", sorted(os.listdir(DATA_DIR)))

frames = []
for name in sorted(os.listdir(DATA_DIR)):
    if name.endswith('.txt'):
        part = pd.read_csv(
            os.path.join(DATA_DIR, name),
            sep=r'\s+', header=None,
            names=['frame', 'ped_id', 'x', 'y'],
        )
        part['scene'] = name.replace('.txt', '')
        frames.append(part)

df = pd.concat(frames, ignore_index=True)
df['ped_id'] = df['ped_id'].astype(int)
df['frame'] = df['frame'].astype(int)

print("Shape (rows, columns):", df.shape)
print("Scenes:", df['scene'].nunique())
print(df.dtypes)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files found: ['biwi_eth.txt', 'biwi_hotel.txt', 'crowds_zara01.txt', 'crowds_zara02.txt', 'crowds_zara03.txt', 'students001.txt', 'students003.txt', 'uni_examples.txt']
Shape (rows, columns): (74428, 5)
Scenes: 8
frame       int64
ped_id      int64
x         float64
y         float64
scene      object
dtype: object


,frame,ped_id,x,y,scene
0,780,1,8.46,3.59,biwi_eth
1,790,1,9.57,3.79,biwi_eth
2,800,1,10.67,3.99,biwi_eth
3,800,2,13.64,5.80,biwi_eth
4,810,1,11.73,4.32,biwi_eth


In [23]:
print(df.isna().sum())
print()
print("Duplicate (scene, frame, ped_id) rows:", df.duplicated(['scene', 'frame', 'ped_id']).sum())
print()
print("Coordinate ranges per scene:")
print(df.groupby('scene')[['x', 'y']].agg(['min', 'max']))

frame     0
ped_id    0
x         0
y         0
scene     0
dtype: int64

Duplicate (scene, frame, ped_id) rows: 0

Coordinate ranges per scene:
                      x                     y           
                    min        max        min        max
scene                                                   
biwi_eth      -7.690000  14.420000  -3.170000  13.210000
biwi_hotel    -3.250000   4.350000 -10.310000   4.310000
crowds_zara01 -0.139538  15.480551  -0.374696  12.386444
crowds_zara02 -0.357791  15.558423  -0.273743  13.942744
crowds_zara03 -0.342006  15.563895  -0.067779  12.714362
students001   -0.461971  15.469186  -0.318372  13.891910
students003   -0.174686  15.436984  -0.222192  13.854201
uni_examples  -0.428717  15.613144   0.032219  13.858975


In [24]:
# Track length: how many points each pedestrian has
lengths = df.groupby(['scene', 'ped_id']).size()
print(lengths.describe())
print("Tracks shorter than 20 points:", (lengths < 20).sum(), "of", len(lengths))

count    2205.000000
mean       33.754195
std        36.320843
min         2.000000
25%        16.000000
50%        26.000000
75%        38.000000
max       584.000000
dtype: float64
Tracks shorter than 20 points: 784 of 2205
